# Jon2 AI — Controlled Arabic Training Experiment

تدريب بحثي باستخدام الدفعة 01 فقط، وهي الدفعة الوحيدة التي صدرت بشأنها موافقة صريحة على التدريب. الدفعتان 02 و03 مستبعدتان حتى صدور موافقة منفصلة. لا تُستخدم محادثات خاصة أو مجموعات بيانات خارجية. هذه تجربة صغيرة وليست نموذجًا بمستوى الأنظمة التجارية الكبيرة.


In [ ]:
import platform, sys, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('RAM check follows')
try:
 import psutil
 print(f'RAM total: {psutil.virtual_memory().total / 1024**3:.2f} GB')
except Exception as exc:
 print('RAM unavailable:', type(exc).__name__)


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, urllib.request

# Download pinned, project-owned source files from the reviewed repository commit.
COMMIT = "550afa949d56dbf81d1909c444a70b3eb0ce6ca6"
BASE = f"https://raw.githubusercontent.com/jonkabeel-star/jon2-ai/{COMMIT}/"
ROOT = Path("/kaggle/working/jon2-source")
if ROOT.exists():
    shutil.rmtree(ROOT)
ENGINE = ROOT / "ai_engine"
TRAINING = ENGINE / "training_data"
TRAINING.mkdir(parents=True, exist_ok=True)

FILES = [
    "ai_engine/jon2_core.py",
    "ai_engine/train_jon2.py",
    "ai_engine/tokenizer.py",
    "ai_engine/source_approval.py",
    "ai_engine/training_data/original_dialogue_ar_batch_01.txt",
]
for relative in FILES:
    destination = ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(BASE + relative, destination)

# Verify the previously approved batch 01 by its exact Git blob fingerprint.
source = TRAINING / "original_dialogue_ar_batch_01.txt"
raw = source.read_bytes()
git_blob = hashlib.sha1(b"blob " + str(len(raw)).encode() + bytes([0]) + raw).hexdigest()
EXPECTED_GIT_BLOB = "9695751952c50a395735376632b3544bcb4b8fe3"
if git_blob != EXPECTED_GIT_BLOB:
    raise RuntimeError(f"Approved batch 01 fingerprint mismatch: {git_blob}")

# Only these three project-authored dialogue batches enter the training corpus.
APPROVED = [
    "original_dialogue_ar_batch_01.txt",
]
DATA = Path("/kaggle/working/jon2-approved-data")
if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)
records = []
for name in APPROVED:
    raw_text = (TRAINING / name).read_bytes()
    decoded = raw_text.decode("utf-8", errors="strict")
    if len(decoded.strip()) < 1000:
        raise RuntimeError(f"Approved batch is unexpectedly short: {name}")
    (DATA / name).write_bytes(raw_text)
    records.append({
        "file": name,
        "sha256": hashlib.sha256(raw_text).hexdigest(),
        "character_count": len(decoded),
        "status": "APPROVED",
        "reviewer": "Repository owner — explicit approval for batch 01 research training only",
        "review_date": "2026-11-04",
        "rights_basis": "Original project-authored Arabic dialogue, created for this project. No private conversations or copied third-party dataset text. This records owner approval for research training only, not a legal guarantee.",
        "intended_use": "research_training"
    })
(DATA / "source_approvals.json").write_text(
    json.dumps({"approved_sources": records}, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Pinned source commit:", COMMIT)
print("Verified batch 01 Git blob:", git_blob)
print("Approved training files:", APPROVED)
print("Approved corpus characters:", sum(item["character_count"] for item in records))
assert len(list(DATA.glob("*.txt"))) == len(records) == 1
assert [record["file"] for record in records] == ["original_dialogue_ar_batch_01.txt"]


In [ ]:
import os, subprocess
ENGINE = ROOT / 'ai_engine'
BEST = Path('/kaggle/working/jon2-checkpoints/jon2-small.pt')
LAST = Path('/kaggle/working/jon2-checkpoints/jon2-last.pt')
BEST.parent.mkdir(parents=True,exist_ok=True)
command = [sys.executable,str(ENGINE/'train_jon2.py'),'--data',str(DATA),'--approval-manifest',str(DATA/'source_approvals.json'),'--output',str(BEST),'--last-checkpoint',str(LAST),'--steps','1200','--batch-size','8','--context','96','--width','96','--heads','4','--layers','3','--warmup-steps','20','--eval-steps','8','--eval-every','100','--patience','8','--minimum-chars','100000','--tokenizer-vocab-size','768','--keep-last']
print('Starting controlled training:', ' '.join(command))
subprocess.run(command,cwd=str(ENGINE),check=True)
if not BEST.is_file() or BEST.stat().st_size < 10000:
 raise RuntimeError('A valid best checkpoint was not produced.')
print('TRAINING CHECKPOINT CREATED:', BEST, BEST.stat().st_size, 'bytes')


In [ ]:
import sys, torch
sys.path.insert(0,str(ENGINE))
from jon2_core import load_model, generate
model, tokenizer, device = load_model(str(BEST))
state = torch.load(BEST,map_location='cpu',weights_only=False)
steps = int(state.get('training_steps',0))
if steps <= 0: raise RuntimeError('Checkpoint reports zero training steps.')
prompts = [
 'المستخدم: ما الفرق بين البيانات والمعلومات؟\nJon2:',
 'المستخدم: احسب 15 بالمئة من 200.\nJon2:',
 'المستخدم: كيف أتحقق من صحة خبر؟\nJon2:',
 'المستخدم: ما وظيفة نظام التشغيل؟\nJon2:',
 'المستخدم: كيف أتعامل مع خطأ في برنامج؟\nJon2:',
 'المستخدم: لماذا لا يثبت الارتباط السببية؟\nJon2:',
 'المستخدم: ما الفرق بين الكتلة والوزن؟\nJon2:',
 'المستخدم: كيف أحمي كلمة المرور؟\nJon2:'
]
print('Checkpoint load: PASS')
print('Recorded training steps:',steps)
print('Corpus characters:',state.get('corpus_characters'))
print('Validation loss:',state.get('best_validation_loss'))
for prompt in prompts:
 sample = generate(model,tokenizer,prompt,device,max_new_chars=140,temperature=0.65,stop_sequences=['\nالمستخدم:'])
 answer = sample[len(prompt):].strip() if sample.startswith(prompt) else sample.strip()
 print('\nPROMPT:',prompt.split('\n')[0])
 print('ANSWER:',answer)
 if not answer: raise RuntimeError('Generation returned an empty answer.')
print('NOTE: outputs require human quality review; successful generation does not prove assistant-level quality.')


## حدود التجربة

هذه تجربة تدريب أصلية صغيرة وليست نموذجًا بمستوى ChatGPT أو Gemini. لا تُفعّل واجهة الموقع تلقائيًا ولا تنشر الأوزان للعامة. يلزم تقييم بشري على مجموعة اختبار مستقلة، ثم إعداد استضافة FastAPI منفصلة قبل تشغيل المحادثة على الموقع.